# 04 - RQ3: Full vs. Reduced Feature Set

**Notebook version:** v13 -- 2026-07-29

Tests whether a SHAP-guided reduced feature set (~10 features, per the
RQ1 events-per-variable constraint) matches the full 591-feature model's
performance, without the two pitfalls flagged during synopsis review:

1. **Information leakage** - SHAP importance is computed fold-by-fold on
   training data only, never on the full dataset before splitting.
2. **Single-method bias** - an independent LASSO-based selection cross-checks
   the SHAP-selected set; agreement between the two is evidence of real signal.

See `docs/synopsis.docx`, "Solution to RQ3" for the full write-up, and
`src/feature_selection.py` for the underlying implementation.

In [ ]:
# --- Colab setup: run this cell first if you opened this notebook from GitHub in Colab ---
# If you're running locally in Jupyter from the notebooks/ folder, this cell does nothing.
# Safe to re-run: always anchors to /content so repeated runs never create nested clones.
# git pull always runs (cheap, ~seconds) so code is never stale even if Colab's
# 'Restart session' left /content on disk from an earlier session -- only
# pip install (the actual slow part) is skipped via the session marker.
import os
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/WJPsystems/secom-explainable-vm.git"
REPO_NAME = "secom-explainable-vm"
SETUP_MARKER = Path("/content/.secom_setup_done")

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists(REPO_NAME):
        !git clone "{REPO_URL}"
        SETUP_MARKER.unlink(missing_ok=True)  # fresh clone -- force full setup below

    # Always pull -- cheap, and guarantees code is current even if /content
    # persisted on disk from an earlier session (e.g. Colab 'Restart session'
    # rather than a full 'Disconnect and delete runtime').
    os.chdir(f"/content/{REPO_NAME}")
    !git pull
    os.chdir("/content")

    os.chdir(f"/content/{REPO_NAME}/notebooks")

    already_setup = SETUP_MARKER.exists()
    if not already_setup:
        !pip install -q -r ../requirements.txt
        SETUP_MARKER.touch()
        setup_note = "Ran pip install (git pull always runs regardless)."
    else:
        setup_note = "Skipped pip install -- already done earlier this session. git pull always ran above."

    commit_info = subprocess.run(
        ["git", "log", "-1", "--format=%h %ci"], capture_output=True, text=True
    ).stdout.strip()
    print(f"Colab setup complete. Working directory: {os.getcwd()}")
    print(setup_note)
    print(f"Repo commit: {commit_info}")
    print("Compare this commit hash against GitHub's latest commit to confirm you're current.")
else:
    print("Not running in Colab -- assuming local Jupyter launched from the notebooks/ folder.")

In [ ]:
import sys

# Resolve src/ absolutely, independent of cell run order or current working
# directory -- works whether or not the Colab setup cell above has run yet.
try:
    import google.colab
    _SRC_PATH = "/content/secom-explainable-vm/src"
except ImportError:
    _SRC_PATH = "../src"
if _SRC_PATH not in sys.path:
    sys.path.append(_SRC_PATH)

import pandas as pd
from xgboost import XGBClassifier

from preprocessing import load_raw, screen_missingness, screen_variance, impute_median
from feature_selection import (
    nested_cv_shap_selection,
    lasso_cross_check,
    agreement_report,
)

X, y = load_raw()
X = impute_median(screen_variance(screen_missingness(X)))
print(f"Screened feature count: {X.shape[1]}")

## Step 1: Nested-CV SHAP feature selection

For each fold: cluster correlated sensors, fit the model on training data
only, compute SHAP on that same training data, and select the top-10
cluster representatives. Report how stable the selection is across folds -
this stability number is itself a finding worth including in the write-up.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from artifacts import load_json

# Use RQ1's actual best tree ensemble architecture (not a hardcoded guess) --
# nested CV needs to retrain fresh per fold, so this rebuilds the same model
# TYPE RQ1 found best, rather than reusing RQ1's already-fitted instance.
rq1_summary = load_json("rq1_summary")
best_tree_name = rq1_summary["best_tree_model_name"]
pos_weight = (y == 0).sum() / (y == 1).sum()

if best_tree_name == "XGBoost":
    model_factory = lambda: XGBClassifier(
        n_estimators=200, max_depth=4, scale_pos_weight=pos_weight,
        eval_metric='logloss', random_state=42, n_jobs=-1,
    )
elif best_tree_name == "RandomForest":
    model_factory = lambda: RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1,
    )
else:
    raise ValueError(f"Unexpected best_tree_model_name from RQ1: {best_tree_name}")

print(f"Using RQ1's best tree ensemble architecture for nested-CV selection: {best_tree_name}")

result = nested_cv_shap_selection(X, y, model_factory, k=10, n_folds=5)

print("\nPer-fold selected features:")
for i, feats in enumerate(result['per_fold_features']):
    print(f"  Fold {i+1}: {feats}")

print("\nFeature stability (how many of 5 folds selected each feature):")
for feat, count in result['feature_stability'].most_common(15):
    print(f"  {feat}: {count}/5")


## Step 2: Build the final feature set and the watch list

The final ~10 features are the most stable across folds (not just the
single best fold). The watch list is the next-ranked 20-30 features that
narrowly missed the cut - reported alongside the model, not discarded.

In [ ]:
final_features = [feat for feat, _ in result['feature_stability'].most_common(10)]
watch_list = sorted(set(f for fold in result['watch_list_by_fold'] for f in fold))

print(f"Final feature set ({len(final_features)}): {final_features}")
print(f"\nWatch list ({len(watch_list)} candidates): {watch_list[:30]}")

## Step 3: Independent LASSO cross-check

A methodologically different feature-selection approach (L1-regularized
logistic regression). High overlap with the SHAP-selected set is
corroborating evidence; low overlap is a flag to investigate before
trusting the reduced set.

In [ ]:
lasso_features = lasso_cross_check(X, y)
print(f"LASSO-selected features ({len(lasso_features)}): {lasso_features[:20]}")

report = agreement_report(final_features, lasso_features[:10])
print(f"\nOverlap: {report['overlap']}")
print(f"SHAP-only: {report['shap_only']}")
print(f"LASSO-only: {report['lasso_only']}")
print(f"Jaccard similarity: {report['jaccard_similarity']:.2f}")

## Step 4: Full vs. reduced model comparison

Compare AUC-ROC (DeLong's test) and accuracy (Cohen's h / two-proportion
test) between the full 591-feature model and the reduced ~10-feature
model, plus McNemar's test on paired prediction disagreements.

In [ ]:
from artifacts import load_json as load_holdout_json
from metrics import bootstrap_auc_comparison, mcnemar_test, cohens_h_two_proportion_test

# Reuse RQ1's fixed 80/20 holdout split -- the SAME split 03 uses for its
# permutation importance -- so full-vs-reduced comparisons here rest on
# genuinely held-out data, not data either model was fit on.
holdout_split = load_holdout_json("rq1_holdout_split")
train_idx, test_idx = holdout_split["train_idx"], holdout_split["test_idx"]
X_train_ho, X_test_ho = X.iloc[train_idx], X.iloc[test_idx]
y_train_ho, y_test_ho = y.iloc[train_idx], y.iloc[test_idx]

# Full-feature model: same architecture as RQ1's best tree, trained on all
# 432 screened features. Reduced model: identical architecture, trained
# only on final_features (the ~10 SHAP-stable features from Step 2).
full_model = model_factory()
full_model.fit(X_train_ho, y_train_ho)
full_probs = full_model.predict_proba(X_test_ho)[:, 1]
full_preds = (full_probs >= 0.5).astype(int)

reduced_model = model_factory()
reduced_model.fit(X_train_ho[final_features], y_train_ho)
reduced_probs = reduced_model.predict_proba(X_test_ho[final_features])[:, 1]
reduced_preds = (reduced_probs >= 0.5).astype(int)

print(f"Full model ({X_train_ho.shape[1]} features) vs. reduced model ({len(final_features)} features)")
print(f"Evaluated on {len(test_idx)} held-out wafers.\n")

# AUC-ROC comparison (DeLong's-test substitute -- see metrics.py docstring
# for why a paired bootstrap is used instead of DeLong's analytic formula)
auc_result = bootstrap_auc_comparison(y_test_ho, full_probs, reduced_probs, n_bootstrap=2000)
print("AUC-ROC comparison (paired bootstrap, DeLong's-test substitute):")
print(f"  Full model AUC-ROC:    {auc_result['auc_a']:.4f}")
print(f"  Reduced model AUC-ROC: {auc_result['auc_b']:.4f}")
print(f"  Difference: {auc_result['observed_diff']:.4f}, p = {auc_result['p_value']:.4f}")

# Accuracy comparison (Cohen's h + two-proportion z-test)
full_acc = (full_preds == y_test_ho).mean()
reduced_acc = (reduced_preds == y_test_ho).mean()
h_result = cohens_h_two_proportion_test(full_acc, len(test_idx), reduced_acc, len(test_idx))
print(f"\nAccuracy comparison (Cohen's h + two-proportion z-test):")
print(f"  Full model accuracy:    {full_acc:.4f}")
print(f"  Reduced model accuracy: {reduced_acc:.4f}")
print(f"  Cohen's h = {h_result['cohens_h']:.4f}, p = {h_result['p_value']:.4f}")

# McNemar's test on paired prediction disagreements
mcnemar_result = mcnemar_test(y_test_ho, full_preds, reduced_preds)
print(f"\nMcNemar's test on paired disagreements ({mcnemar_result['method']}):")
print(f"  Full-right/reduced-wrong: {mcnemar_result['b']}, Full-wrong/reduced-right: {mcnemar_result['c']}")
print(f"  p = {mcnemar_result['p_value']:.4f}")

print(
    "\nInterpretation: if none of the above are significant, the reduced "
    "model is not measurably worse than the full model on this held-out "
    "set -- supporting RQ3's hypothesis. A significant result would mean "
    "the feature reduction cost real predictive performance, which is worth "
    "reporting honestly either way."
)


## Next steps

- Feed `final_features` into `05_attention_comparison_rq4.ipynb`
- Report feature stability, watch list, and LASSO overlap in the capstone write-up
- See `06_anomaly_safety_net.ipynb` for the exploratory full-feature-set safety net

## Export

Run the cell below last to export this notebook to a standalone HTML file.

In [ ]:
# --- Export this notebook to HTML (run this cell last) ---
# Works whether opened live from GitHub in Colab or run locally in Jupyter.
import json
import subprocess

NOTEBOOK_NAME = "04_feature_reduction_rq3"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

export_path = f"{NOTEBOOK_NAME}.ipynb"
live_export_available = False

if IN_COLAB:
    # Colab's own notebook JSON isn't the same file as the clone on disk --
    # this pulls the live, currently-run state (including your outputs)
    # directly from the Colab frontend, so nothing gets missed. This only
    # works when there's an actual live frontend attached (i.e. you're running
    # this cell interactively yourself) -- it returns None instead of raising
    # when run unattended (e.g. via 00_run_all.ipynb's automated execution),
    # so that case is caught explicitly here rather than left to crash with a
    # raw TypeError, which used to make 00_run_all's --allow-errors flag mask
    # *real* failures elsewhere in the notebook, not just this expected one.
    from google.colab import _message
    response = _message.blocking_request('get_ipynb', timeout_sec=30)
    if response is not None:
        ipynb_content = response['ipynb']
        with open(export_path, 'w') as f:
            json.dump(ipynb_content, f)
        live_export_available = True
    else:
        print(
            "No live Colab frontend detected (expected when run via "
            "00_run_all.ipynb) -- skipping the live export. 00_run_all does its "
            "own separate HTML export against the already-executed file instead."
        )

if live_export_available or not IN_COLAB:
    html_output = f"{NOTEBOOK_NAME}.html"
    result = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', export_path, '--output', html_output],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    else:
        print(f"Exported to {html_output}")

    if IN_COLAB and result.returncode == 0:
        from google.colab import files
        files.download(html_output)
